In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

# ==========================================================
# LOAD MODEL
# ==========================================================
model_huruf = joblib.load("model_alfabet.pkl")
model_angka = joblib.load("model_angka.pkl")
model_kata  = joblib.load("model_kata.pkl")

# ==========================================================
# LABEL KATA
# Model hanya mengeluarkan angka 0-6, lalu dikonversi ke kata di sini
# ==========================================================
LABEL_KATA = {
    0: "berdoa",
    1: "berjalan",
    2: "berpikir",
    3: "makan",
    4: "mandi",
    5: "saya",
    6: "tidur",
}

# ==========================================================
# MEDIAPIPE SETUP MEDIAPIPE HOLISTIC
# Holistic dipakai karena mode kata butuh tangan + wajah + pose sekaligus
# Mode huruf & angka sebenarnya cukup pakai Hands, tapi kita pakai
# Holistic agar cukup 1 proses untuk semua mode
# ==========================================================
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ==========================================================
# LANDMARK INDEX
# Titik-titik wajah dan pose yang dipakai sebagai fitur
# ==========================================================
FACE_POINTS = [1, 33, 61, 199, 263] # Hidung, mata kiri, mulut kiri, dagu, mata kanan
POSE_POINTS = [11, 12, 13, 14, 15]  # Bahu kiri, bahu kanan, siku kiri, siku kanan, pergelangan

# ==========================================================
# MODE
# ==========================================================
mode = "huruf" # Default saat program pertama dibuka

# ==========================================================
# CAMERA
# VideoCapture(0) = kamera utama/webcam laptop
# ==========================================================
cap = cv2.VideoCapture(0)

# Loop utama — terus berjalan sampai user menekan 'Q'
while True:

    ret, frame = cap.read() # Ambil 1 frame dari kamera
    if not ret:
        break

    frame = cv2.flip(frame, 1)
     # Konversi BGR → RGB karena MediaPipe butuh format RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Proses frame dengan MediaPipe — hasilkan semua landmark tangan, wajah, pose
    result = holistic.process(rgb)
    pred = "-"

    # ======================================================
    # MODE HURUF / ANGKA
    # ======================================================
    if mode in ["huruf", "angka"]:
        # Coba ambil tangan kanan dulu, kalau tidak ada pakai tangan kiri
        hand = result.right_hand_landmarks or result.left_hand_landmarks

        if hand: # Hanya proses jika ada tangan yang terdeteksi

            # =========================
            # HURUF
            # Kedua mode ini hanya butuh landmark tangan (21 titik)
            # =========================
            if mode == "huruf":
                data = []
                for lm in hand.landmark:
                    data.append(lm.x)
                    data.append(lm.y)

                if len(data) == 42: # Pastikan semua 21 titik terdeteksi
                    data = np.array(data).reshape(1, -1)
                    pred = str(model_huruf.predict(data)[0]) # Prediksi huruf

            # =========================
            # ANGKA
            # =========================
            elif mode == "angka":

                lm_list = hand.landmark
                data = []
                # Koordinat pergelangan tangan sebagai titik acuan (origin)
                base_x = lm_list[0].x
                base_y = lm_list[0].y

                for lm in lm_list:
                    data.append(lm.x - base_x)
                    data.append(lm.y - base_y)
                    
                # Normalisasi ke rentang [0, 1]
                data = np.array(data)
                data = data - np.min(data)

                if np.max(data) != 0:
                    data = data / np.max(data)
                # Tambahkan fitur jarak antar ujung jari (sama dengan saat training)
                data = data.tolist()
                 # Tambahkan fitur jarak antar ujung jari (sama dengan saat training)
                def dist(a, b):
                    return ((a.x - b.x)**2 + (a.y - b.y)**2) ** 0.5

                data.append(dist(lm_list[4], lm_list[8])) # Ibu jari → telunjuk
                data.append(dist(lm_list[8], lm_list[12])) # Telunjuk → tengah
                data.append(dist(lm_list[12], lm_list[16])) # Tengah → manis
                data.append(dist(lm_list[16], lm_list[20])) # Manis → kelingking

                data = np.array(data).reshape(1, -1)
                pred = str(model_angka.predict(data)[0])

    elif mode == "kata":
    
        data = []
        # normalisasi koordinat landmark (relatif + scale)
        def normalize_landmarks(landmarks):
            pts = np.array([[lm.x, lm.y] for lm in landmarks])
            base = pts[0] # Titik 0 sebagai acuan (pergelangan tangan)
            pts = pts - base # Jadikan koordinat relatif
    
            max_val = np.max(np.abs(pts))
            if max_val != 0:
                pts = pts / max_val # Normalisasi skala
    
            return pts.flatten().tolist()
        # hitung jarak antar ujung jari
        def pairwise_distances(landmarks):
            pairs = [(4,8), (8,12), (12,16), (16,20)]
            dist = []
            for a,b in pairs:
                dx = landmarks[a].x - landmarks[b].x
                dy = landmarks[a].y - landmarks[b].y
                dist.append((dx**2 + dy**2)**0.5)
            return dist
    
        # =========================
        # LEFT HAND
        # =========================
        if result.left_hand_landmarks:
            left = result.left_hand_landmarks.landmark
            data += normalize_landmarks(left)
            data += pairwise_distances(left)
        else:
            data += [0.0] * (42 + 4)
    
        # =========================
        # RIGHT HAND
        # =========================
        if result.right_hand_landmarks:
            right = result.right_hand_landmarks.landmark
            data += normalize_landmarks(right)
            data += pairwise_distances(right)
        else:
            data += [0.0] * (42 + 4)
    
        # =========================
        # FACE
        # =========================
        if result.face_landmarks:
            face = result.face_landmarks.landmark
            base = np.array([face[1].x, face[1].y]) # Hidung sebagai acuan
    
            for i in FACE_POINTS:
                lm = face[i]
                pt = np.array([lm.x, lm.y]) - base
                data += pt.tolist()
        else:
            data += [0.0] * 10
    
        # =========================
        # POSE
        # =========================
        if result.pose_landmarks:
            pose = result.pose_landmarks.landmark
            base = np.array([pose[11].x, pose[11].y]) # Bahu kiri sebagai acuan
    
            for i in POSE_POINTS:
                lm = pose[i]
                pt = np.array([lm.x, lm.y]) - base
                data += pt.tolist()
        else:
            data += [0.0] * 10
    
        # =========================
        # EXTRA (SAMA DENGAN TRAINING)
         # Jarak tangan kanan ke wajah — penting untuk kata yang melibatkan
        # =========================
        extra = []
    
        def dist_xy(a, b):
            return ((a.x - b.x)**2 + (a.y - b.y)**2) ** 0.5
    
        if result.right_hand_landmarks and result.face_landmarks:
            hand = result.right_hand_landmarks.landmark
            face = result.face_landmarks.landmark
    
            extra.append(dist_xy(hand[8], face[1])) # Ujung telunjuk → hidung
            extra.append(dist_xy(hand[4], face[1]))  # Ujung ibu jari → hidung
        else:
            extra += [0.0, 0.0]
    
        data += extra
    
        # =========================
        # PREDIKSI
        # =========================
        if len(data) == 114:
            data = np.array(data).reshape(1, -1)
            pred_index = int(model_kata.predict(data)[0]) # Model keluarkan angka 0-6
            pred = LABEL_KATA.get(pred_index, f"?{pred_index}") # Konversi ke nama kata
        else:
            pred = f"err:{len(data)}"

    # ======================================================
    # TAMPILAN
    # ======================================================
    cv2.putText(frame, f"MODE  : {mode.upper()}",
                (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, (0, 255, 0), 2)

    cv2.putText(frame, f"HASIL : {pred}",
                (10, 80),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, (0, 255, 255), 2)

    cv2.putText(frame,
                "A=Huruf  N=Angka  K=Kata  Q=Keluar",
                (10, 460),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6, (255, 255, 255), 2)

    cv2.imshow("Final Project Sign Language", frame)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('a'):
        mode = "huruf"
    elif key == ord('n'):
        mode = "angka"
    elif key == ord('k'):
        mode = "kata"
    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [5]:
import os
import joblib

model_kata = joblib.load("model_kata.pkl")
print("Classes:", model_kata.classes_)

# Cek nama-nama kata unik dari folder images
folder = "Dataset_Kata/train/images"  # sesuaikan path
namafile = os.listdir(folder)

# Ambil kata pertama dari nama file (sebelum tanda '-')
kata_unik = sorted(set(f.split('-')[0] for f in namafile))
print("Kata yang ada:", kata_unik)

Classes: [0 1 2 3 4 5 6]
Kata yang ada: ['berdoa', 'berjalan', 'berpikir', 'makan', 'mandi', 'saya', 'tidur']


In [3]:
import cv2
import mediapipe as mp
import os
import csv
import numpy as np
import random

DATASET_PATH = "dataset_angka"
OUTPUT = "angka3.csv"
TARGET_PER_CLASS = 400 

# Inisialisasi modul deteksi tangan dari MediaPipe
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.6  # minimum keyakinan deteksi = 60%
)

# ==========================================================
# AUGMENTASI
# ==========================================================
def augment_image(img):
    h, w = img.shape[:2]
    aug = img.copy()

    # 🔹 brightness & contrast 
    alpha = random.uniform(0.9, 1.1)
    beta  = random.randint(-15, 15)
    aug = cv2.convertScaleAbs(aug, alpha=alpha, beta=beta)

    # 🔹 rotasi kecil
    angle = random.uniform(-10, 10)
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    aug = cv2.warpAffine(aug, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    # 🔹 zoom ringan
    scale = random.uniform(0.95, 1.0)
    cx, cy = w // 2, h // 2
    nw, nh = int(w * scale), int(h * scale)
    x1 = max(cx - nw // 2, 0)
    y1 = max(cy - nh // 2, 0)
    x2 = min(x1 + nw, w)
    y2 = min(y1 + nh, h)
    aug = cv2.resize(aug[y1:y2, x1:x2], (w, h))

    return aug

# ==========================================================
# EKSTRAK LANDMARK 
# ==========================================================
def extract_landmarks(rgb_img):
    result = hands.process(rgb_img)

    if not result.multi_hand_landmarks:
        return None

    landmarks = result.multi_hand_landmarks[0]

    # pakai RELATIVE POSITION 
    base_x = landmarks.landmark[0].x
    base_y = landmarks.landmark[0].y

    data = []

    for lm in landmarks.landmark:
        data.append(lm.x - base_x)
        data.append(lm.y - base_y)

    # NORMALISASI
    data = np.array(data)
    data = data - np.min(data)
    if np.max(data) != 0:
        data = data / np.max(data)

    # FILTER DATA JELEK
    if np.max(data) - np.min(data) < 0.05:
        return None

    # TAMBAHAN FITUR JARAK JARI
    lm = landmarks.landmark
    def dist(a, b):
        return ((a.x - b.x)**2 + (a.y - b.y)**2) ** 0.5

    data = data.tolist()
    data.append(dist(lm[4], lm[8]))
    data.append(dist(lm[8], lm[12]))
    data.append(dist(lm[12], lm[16]))
    data.append(dist(lm[16], lm[20]))

    return data

# ==========================================================
# MAIN LOOP UTAMA PENGUMPULAN DATA
# ==========================================================
with open(OUTPUT, 'w', newline='') as f:
    writer = csv.writer(f)

    for label in sorted(os.listdir(DATASET_PATH)):
        folder = os.path.join(DATASET_PATH, label)
        if not os.path.isdir(folder):
            continue

        img_files = [
            os.path.join(folder, fn)
            for fn in os.listdir(folder)
            if fn.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
        ]

        if not img_files:
            print(f"[{label}] tidak ada gambar")
            continue

        print(f"[{label}] mulai dari {len(img_files)} gambar")

        # 🎯 balance ringan
        if str(label) == "6":
            TARGET = 200
        else:
            TARGET = TARGET_PER_CLASS

        berhasil = 0
        gagal = 0
        attempts = 0
        MAX_ATTEMPTS = TARGET * 10

        while berhasil < TARGET and attempts < MAX_ATTEMPTS:
            attempts += 1

            src_path = random.choice(img_files)
            img = cv2.imread(src_path)
            if img is None:
                continue

            img = cv2.resize(img, (640, 480))

            #augme ntasi lebih seimbang
            if random.random() < 0.5:
                processed = img.copy()
            else:
                processed = augment_image(img)

            rgb = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)
            data = extract_landmarks(rgb)

            if data:
                data.append(label)
                writer.writerow(data)
                berhasil += 1

                if berhasil % 50 == 0:
                    print(f"   {berhasil}/{TARGET}")
            else:
                gagal += 1

        print(f"[{label}] ✅ {berhasil}/{TARGET} | gagal: {gagal} | attempts: {attempts}\n")

hands.close()
print("✅ selesai:", OUTPUT)

[0] mulai dari 56 gambar
   50/400
   100/400
   150/400
   200/400
   250/400
   300/400
   350/400
   400/400
[0] ✅ 400/400 | gagal: 327 | attempts: 727

[1] mulai dari 56 gambar
   50/400
   100/400
   150/400
   200/400
   250/400
   300/400
   350/400
   400/400
[1] ✅ 400/400 | gagal: 25 | attempts: 425

[2] mulai dari 56 gambar
   50/400
   100/400
   150/400
   200/400
   250/400
   300/400
   350/400
   400/400
[2] ✅ 400/400 | gagal: 44 | attempts: 444

[3] mulai dari 56 gambar
   50/400
   100/400
   150/400
   200/400
   250/400
   300/400
   350/400
   400/400
[3] ✅ 400/400 | gagal: 0 | attempts: 400

[4] mulai dari 56 gambar
   50/400
   100/400
   150/400
   200/400
   250/400
   300/400
   350/400
   400/400
[4] ✅ 400/400 | gagal: 2 | attempts: 402

[5] mulai dari 56 gambar
   50/400
   100/400
   150/400
   200/400
   250/400
   300/400
   350/400
   400/400
[5] ✅ 400/400 | gagal: 0 | attempts: 400

[6] mulai dari 56 gambar
   50/200
   100/200
   150/200
   200/200
[6] 

In [2]:
import joblib

model_huruf = joblib.load("model_alfabet.pkl")
model_angka = joblib.load("model_angka.pkl")
model_kata  = joblib.load("model_kata.pkl")

print(type(model_huruf))
print(type(model_angka))
print(type(model_kata))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
<class 'sklearn.ensemble._forest.RandomForestClassifier'>
<class 'sklearn.ensemble._forest.RandomForestClassifier'>
